# Setting parameters

In this tutorial, we will explain how to set the different parameters using the `SetupParams` class. We will explore the role of each variable and how they are interconnected. While the package includes several safeguards to ensure that the assigned parameter values are physically meaningful, it is ultimately the user's responsibility to ensure that each parameter is physically plausible. Failing to do so may lead to unreliable results.

#### Plan of this notebook
1. Brief introduction to the different parameters
2. How to set the parameters: fixing the number of non-vacuum pulses, the non-vacuum probability, or the multi-photon probability
3. Calculating the SNR, signal rate, and noise rate

After this short tutorial, you will be able to initialize the `SetupParams` class, set the different parameters, and compute the signal rate, noise rate, and SNR of different sources. With this knowledge, you will be ready to move on to the other tutorials provided with the package.


## 1) Brief introduction to the different parameters

The module is designed to model quantum lidars based on three different types of sources:

* **Attenuated pulsed laser:** coherent state
* **Array of single-photon sources (SPSs):** quantum-dot-based sources
* **Correlated photon-pair source:** SPDC source producing a squeezed state

In order to correctly model a quantum lidar based on these sources, we need to set several parameters. The following is a list of the parameters that can be configured:

### General parameters

* `fock_space_dim`: Dimension of the Fock space. It should be large enough to ensure that no probabilities are truncated. $[-]$

### Source parameters

* `output_power`: Optical power of the source in photon number per second. $[s^{-1}]$
* `multi_photon_probability`: Multi-photon probability of the source. $[-]$
* `no_vacuum_probability`: Non-vacuum probability of the source. $[-]$
* `number_nv_pulse`: Number of non-vacuum pulses to enforce. $[s^{-1}]$
* `sp_collection`: Collection efficiency of the source. $[-]$
* `sp_p1`: Probability that the SPS emits a single photon. $[-]$
* `sp_p2`: Probability that the SPS emits two photons. $[-]$
* `spdc_eps_heralding`: Idler collection efficiency. $[-]$
* `spdc_eps_collection`: Collection efficiency of the EPS. $[-]$

### Detection parameters

* `atmosphere`: Atmospheric transmission efficiency. $[dB/km]$
* `target_distance`: Distance between the source and the target. $[m]$
* `receiver_diameter`: Diameter of the receiver. $[m]$
* `target_albedo`: Albedo of the target. $[-]$
* `optics_transmitter`: Transmission efficiency of the optical system. $[-]$
* `optics_receiver`: Reception efficiency of the optical system. $[-]$
* `detection_efficiency`: Detection efficiency of the detector. $[-]$
* `background`: Background noise detection rate. $[Hz/cm^2]$
* `detector_dark`: Dark count rate of the detector. $[Hz]$
* `timing_window`: Integration time of the detector. $[s]$

In the model, we use the variable $\eta_{\mathrm{signal}}$ to represent the collection efficiency. In the implementation, however, a distinction is made between the collection efficiency of the SPS and that of the EPS.

In [1]:
from Sources import SetupParameters, PulsedLaser, EntangledPhotonSPDC, SinglePhoton

After importing the SetupParameters class, one can use the function help() to get a brief description of the class.

In [2]:
SetupParameters.help()

Here are the parameters you can set:
--- General ---
fock_space_dim: Dimension of the fock space, must be an integer
--- Sources ---
output_power: optical output power of the source in photon per second [s^-1]
multi_photon_probability: Probability of the laser source emitting more than 1 photon [-]
no_vacuum_probability: Probability of the laser source not emitting a photon [-]
number_nv_pulse: Number of non-vacuum pulses per second [s^-1]
sp_collection: Efficiency of the single photon source collection, does not include detector efficiency [-]
sp_p1: Probability that the single photon source emits 1 photons [-]
sp_p2: Probability that the single photon source emits 2 photons [-]
spdc_eps_heralding: Efficiency of the EPS local measurement, includes collection and detector efficiency [-]
spdc_eps_collection: Efficiency of the EPS collection for the sensing photon, does not include detector efficiency [-]
spdc_emission: SPDC average pair photon number [-]
--- Detection ---
target_distanc

## 2) How to set the parameters

Simply put, the `SetupParameters` class acts as an effective dictionary where the different parameters can be specified. However, the following three parameters are mutually exclusive, meaning that only one of them can be fixed:

* Number of non-vacuum pulses
* Non-vacuum probability
* Multi-photon probability

We emphasize that setting one of these parameters automatically sets the others. In the code, you must specify one of these three parameters and leave the other two set to `None`. Otherwise, an error will be raised.

#### Example: setup parameters for a laser

Now, let's define the parameters for an attenuated laser. Since we are not interested in the other sources for now, we can leave the corresponding collection efficiencies set to `None`.

Let's fix the multi-photon probability at 5%. Since the multi-photon probability is fixed, we must leave the non-vacuum probability and the number of non-vacuum pulses set to `None`.

In [3]:
param_laser = SetupParameters(
	fock_space_dim=5,
	output_power=2e6,
	trigger_rate=None,
	multi_photon_probability=0.05,
	no_vacuum_probability=None,
	number_nv_pulse=None,
	sp_collection=None,
	sp_p1=None,
	sp_p2=None,
	spdc_eps_heralding=None,
	spdc_eps_collection=None,
	atmosphere=1,
	target_distance=1,
	receiver_diameter=0.05,
	target_albedo=0.2,
	optics_transmitter=0.8,
	optics_receiver=0.5,
	detection_efficiency=0.5,
	background=2e5,
	detector_dark=200,
	timing_window=0.5e-9,
)

Now, let's pass the parameters to the PulsedLaser class and let's exact the different probabilities.

In [4]:
laser = PulsedLaser(param_laser)

print(f"The Non-vacuum probability of the laser is {laser.no_vacuum_probability*100:.0f}%")
print(f"The multi-photon probability of the laser is {laser.multi_photon_probability*100:.0f}%")

The Non-vacuum probability of the laser is 30%
The multi-photon probability of the laser is 5%


#### Example : setup parameters for an EPS
As expected, the multi-photon probability is 5% and the other probabilities are computed accordingly. Now, we can do something similar for the EPS. In general, it is recommended (but not mandatory) to perform a deepcopy of the parameters in order to avoid any conflict between the different sources.

In [5]:
from copy import deepcopy

param_eps = deepcopy(param_laser)
param_eps["spdc_eps_heralding"] = 0.1
param_eps["spdc_eps_collection"] = 0.2
param_eps["multi_photon_probability"] = None

Now, let's fix the non-vacuum probability of the EPS at 30%.

In [6]:
param_eps["no_vacuum_probability"] = 0.3

eps = EntangledPhotonSPDC(param_eps)

print(f"The Non-vacuum probability of the EPS is {eps.no_vacuum_probability*100:.0f}%")
print(f"The multi-photon probability of the EPS is {eps.multi_photon_probability*100:.0f}%")
print(f"The number of non-vacuum pulse of the SPDC source is {eps.number_nv_pulse/1e6} MHz")

The Non-vacuum probability of the EPS is 30%
The multi-photon probability of the EPS is 9%
The number of non-vacuum pulse of the SPDC source is 1.4 MHz


#### Example: setup parameters for an SPS

The SPS is a little bit trickier because its average number of photons per pulse is intrinsic to the source. It is not a continuous and "sweepable" parameter. Therefore, once the optical power is fixed, both the non-vacuum probability and the multi-photon probability are also fixed.

The easiest way to deal with an SPS is therefore to simply leave these three parameters set to `None`, and the code will determine them automatically. You can still fix one of the three parameters, but it must be set to its expected value; otherwise, an error will be raised.

If unspecified, the `SinglePhoton` object will assume a single-photon source. However, if desired, an array of identical single-photon sources can also be considered.

In [7]:
param_sps = deepcopy(param_laser)
param_sps["fock_space_dim"] = 3
param_sps["sp_collection"] = 0.2
param_sps["sp_p1"] = 0.99
param_sps["sp_p2"] = 0.001
param_sps["multi_photon_probability"] = None

sps = SinglePhoton(param_sps)

print(f"The Non-vacuum probability of the SPS is {sps.no_vacuum_probability*100:.0f}%")
print(f"The multi-photon probability of the SPS is {sps.multi_photon_probability*100:.4f}%")

The Non-vacuum probability of the SPS is 16%
The multi-photon probability of the SPS is 0.0026%


## 3) Calculating the SNR, the signal rate, and the noise rate

Once the parameters are set, one can calculate the SNR, the signal rate $\mathcal{C}$, and the noise rate $\mathcal{C}_{\mathrm{noise}}$. The SNR is defined as follows:

$$
\textbf{SNR} = \frac{\mathcal{C}-\mathcal{C}_{\mathrm{noise}}}{\mathcal{C}_{\mathrm{noise}}}
$$

The signal rate is the number of photons detected per second in the bin associated with the target's time of flight. This includes both the photons returning from the target and the background noise. The noise rate is the number of photons detected per second in the other bins.

It is important to remember that both rates correspond to a bin with a time window $\tau_w$.

Also, note that a warning will be raised regarding the Fock space dimension of the single-photon source. Here, we use a Fock space dimension of five for all sources. However, for a single single-photon source, a maximum of two photons can be emitted, yielding a Fock space dimension of three, since vacuum emission is also included.

Therefore, a Fock space dimension of five is unnecessarily large. To minimize the computation time, we manually set it to the ideal Fock space dimension.

The warning can be disabled by using `SinglePhoton(param, warning_off=True)`.

#### Example : Compute the signal rate of all sources

In [8]:
param = SetupParameters(
	fock_space_dim=5,
	output_power=2e6,
	trigger_rate=None,
	number_nv_pulse=None,
	multi_photon_probability=None,
	no_vacuum_probability=None,
	sp_collection=0.8,
	sp_p1=0.99,
	sp_p2=1e-3,
	spdc_eps_heralding=0.4,
	spdc_eps_collection=0.8,
	atmosphere=1,
	target_distance=1,
	receiver_diameter=0.05,
	target_albedo=0.2,
	optics_transmitter=0.8,
	optics_receiver=0.5,
	detection_efficiency=0.5,
	background=2e5,
	detector_dark=200,
	timing_window=0.5e-9,
)

sps = SinglePhoton(param)
multi_photon_probability = sps.multi_photon_probability
param["multi_photon_probability"] = multi_photon_probability

laser = PulsedLaser(param)
eps = EntangledPhotonSPDC(param)

signal_laser = laser.signal_rate()
signal_eps = eps.signal_rate()
signal_sps = sps.signal_rate()

print(f"The signal rate of the laser is {signal_laser:.2f} photons/s")
print(f"The signal rate of the EPS is {signal_eps:.2f} photons/s")
print(f"The signal rate of the SPS is {signal_sps:.2f} photons/s")

The signal rate of the laser is 51.75 photons/s
The signal rate of the EPS is 13.33 photons/s
The signal rate of the SPS is 32.17 photons/s


C:\Github\QuantumSimulationLiDAR\Sources.py:263: UserWarning: Fock space dimension should be set to 3. This will be corrected.
  warnings.warn(f"Fock space dimension should be set to {2 * number_sps + 1}. This will be corrected.")


#### Example : Compute the noise rate of all sources

In [9]:
noise_laser = laser.noise_rate()
noise_eps = eps.noise_rate()
noise_sps = sps.noise_rate()

print(f"The noise rate of the laser is {noise_laser:.2f} photons/s")
print(f"The noise rate of the EPS is {noise_eps:.2f} photons/s")
print(f"The noise rate of the SPS is {noise_sps:.2f} photons/s")

The noise rate of the laser is 20.51 photons/s
The noise rate of the EPS is 0.37 photons/s
The noise rate of the SPS is 0.93 photons/s


#### Example : Compute the SNR of all sources

In [10]:
snr_laser = laser.signal_to_noise_rate()
snr_eps = eps.signal_to_noise_rate()
snr_sps = sps.signal_to_noise_rate()

print(f"The SNR of the laser is {snr_laser:.2f}")
print(f"The SNR of the EPS is {snr_eps:.2f}")
print(f"The SNR of the SPS is {snr_sps:.2f}")

The SNR of the laser is 1.52
The SNR of the EPS is 35.46
The SNR of the SPS is 33.46


In this tutorial, we have seen how to set the different parameters using the `SetupParameters` class. We have also seen how to compute the signal rate, noise rate, and SNR. With this knowledge, you will be able to move on to the other tutorials provided with the package.